# DATA 266 Lab 1 — Task 3 — Viraat Chaudhary — Team 15

Run this notebook on Colab connected to the lab's **local Docker CUDA runtime**.
The workflow adapts team code authored by Anshika Goel; attribution and module hashes
are in `configs/code_provenance.json`. No teammate checkpoints, predictions or metrics
are included as your results.

Review the choices before launching: 6 residual blocks, nearest-neighbor resize +
convolution, batch 2, identity ratio 0.25 (effective weight 2.5), independent unpaired
sampling with full photo coverage, and a 60-epoch schedule (20 constant + 40 decay).
BF16 is used where supported; parameters and Adam state remain FP32. Score improvement
is a hypothesis to test, not a claimed result. The assignment reserves core design
decisions and analysis for your own understanding; be able to explain these choices.

**Order:** setup → original data → GPU benchmark → launch → status.
The benchmark prints an estimated duration. The long run is launched as a detached
process; the launch cell finishes after a few seconds. Keep the machine and Docker
container running, with a reservation covering the run. Closing Colab does not stop
the detached process, but stopping Docker, logging off if it stops Docker, restarting
or wiping the machine does. Checkpoints remain under the mounted project folder.


## 1. Locate your files and verify the actual GPU


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import zipfile
import torch

# Optional: set this to the folder containing task3_gan. Otherwise locate it.
PROJECT_ROOT = ""
member_relative = Path("task3_gan/Viraat_Chaudhary")
search_bases = [Path("/app"), Path.cwd(), Path("/content")]
candidates = []
if PROJECT_ROOT:
    candidates.append(Path(PROJECT_ROOT))
else:
    for base in search_bases:
        if not base.is_dir():
            continue
        candidates.append(base)
        candidates.extend(p for p in base.iterdir() if p.is_dir() and not p.name.startswith('.'))
REPO_ROOT = next((p.resolve() for p in candidates if (p/member_relative/'src/train.py').is_file()), None)
if REPO_ROOT is None:
    print("Select the small Viraat_Task3_Ready.zip from this chat.")
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError('Extract Viraat_Task3_Ready.zip into the Windows folder mounted to /app, then rerun this cell.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Upload only Viraat_Task3_Ready.zip here; the dataset comes next.")
    archive_name, archive_bytes = next(iter(uploaded.items()))
    import io
    REPO_ROOT = Path('/app') if Path('/app').is_dir() else Path.cwd()
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as z:
        for entry in z.infolist():
            if entry.is_dir():
                continue
            relative = Path(entry.filename)
            if relative.is_absolute() or '..' in relative.parts:
                raise ValueError("Unsafe archive path")
            if not (entry.filename.startswith('task3_gan/Viraat_Chaudhary/') or entry.filename=='START_HERE.md'):
                raise ValueError("This is not the Viraat training ZIP")
            target=REPO_ROOT/relative
            if target.exists():
                raise FileExistsError("Refusing to overwrite existing code: "+str(target))
            z.extract(entry,REPO_ROOT)
    if not (REPO_ROOT/member_relative/'src/train.py').is_file():
        raise RuntimeError("Training files were not found after extracting the ZIP")
    del uploaded,archive_bytes
MEMBER_ROOT = REPO_ROOT/member_relative
SRC = MEMBER_ROOT/'src'
CONFIG = MEMBER_ROOT/'configs/cyclegan_viraat.json'
RUN_ID = "viraat_resizeconv6_run001"
print("Project:",REPO_ROOT)
print("PyTorch:",torch.__version__,"CUDA runtime:",torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable. Reconnect Colab to the lab's Docker local runtime.")
print("GPU:",torch.cuda.get_device_name(0))
print("BF16 supported:",torch.cuda.is_bf16_supported())
print("Run:",RUN_ID)
print("Config:",json.loads(CONFIG.read_text())['training'])


## 2. Load the original shared dataset


In [ ]:
# Use the team's ORIGINAL Monet/photo dataset, never generated predictions.
# If the shared data are already here, this cell just verifies them.
# Otherwise select a ZIP containing monet_jpg and photo_jpg when prompted.
DATA_ZIP_PATH = ""  # Optional existing dataset ZIP inside the runtime.
data_root=REPO_ROOT/'task3_gan/data'
a_dir,b_dir=data_root/'monet_jpg',data_root/'photo_jpg'
if not (a_dir.is_dir() and b_dir.is_dir()):
    if not DATA_ZIP_PATH:
        possible=[]
        for base in {REPO_ROOT,REPO_ROOT.parent,Path('/app')}:
            if not base.is_dir():
                continue
            for candidate in base.glob('*.zip'):
                try:
                    with zipfile.ZipFile(candidate) as archive:
                        names=archive.namelist()
                        if any('/monet_jpg/' in '/'+name for name in names) and any('/photo_jpg/' in '/'+name for name in names):
                            possible.append(candidate.resolve())
                except zipfile.BadZipFile:
                    pass
        possible=list(dict.fromkeys(possible))
        if len(possible)==1:
            DATA_ZIP_PATH=str(possible[0])
            print('Original dataset ZIP found:',DATA_ZIP_PATH)
    if not DATA_ZIP_PATH:
        print("Select the ORIGINAL dataset ZIP containing monet_jpg and photo_jpg.")
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError('Download the original dataset ZIP into the Windows folder mounted to /app, then rerun this cell. Google Drive mounting is not needed.')
        uploaded=files.upload()
        if len(uploaded)!=1:
            raise RuntimeError("Select one dataset ZIP")
        DATA_ZIP_PATH=next(iter(uploaded))
        del uploaded
    with zipfile.ZipFile(DATA_ZIP_PATH) as z:
        for domain in ['monet_jpg','photo_jpg']:
            entries=[]
            for entry in z.infolist():
                p=Path(entry.filename)
                if entry.is_dir() or '__MACOSX' in p.parts or p.name.startswith('.'):
                    continue
                if domain in p.parts and p.suffix.lower() in ['.jpg','.jpeg','.png']:
                    if p.is_absolute() or '..' in p.parts:
                        raise ValueError('Unsafe dataset ZIP')
                    entries.append(entry)
            expected=300 if domain=='monet_jpg' else 7038
            if len(entries)!=expected or len({Path(e.filename).name for e in entries})!=expected:
                raise RuntimeError(f'{domain}: expected {expected} unique original images, found {len(entries)}')
            dest=data_root/domain
            dest.mkdir(parents=True,exist_ok=True)
            for entry in entries:
                target=dest/Path(entry.filename).name
                if target.exists():
                    if target.read_bytes()!=z.read(entry):
                        raise RuntimeError('Existing dataset image differs: '+target.name)
                else:
                    target.write_bytes(z.read(entry))
image_exts={'.jpg','.jpeg','.png','.bmp','.webp'}
counts={domain:len([p for p in (data_root/domain).iterdir() if p.is_file() and p.suffix.lower() in image_exts])
        for domain in ['monet_jpg','photo_jpg']}
print('Original data counts:',counts)
if counts!={'monet_jpg':300,'photo_jpg':7038}:
    raise RuntimeError('Use the same original 300/7038 dataset as the team baseline')
print('Shared data ready. Your sampling and training outputs remain in your own folder.')


## 3. Run the short GPU benchmark

This exercises the real model, losses, optimizer and data loader for 13 batches.
The first 3 are warmup; the remaining 10 estimate speed. These temporary weights
are discarded. Production training still starts from fresh random weights.
If the duration does not fit your allocation, resolve that before the long launch.


In [ ]:
result=subprocess.run([sys.executable,'-u',str(SRC/'lab_tools.py'),'benchmark'],cwd=REPO_ROOT)
if result.returncode:
    raise RuntimeError('GPU benchmark failed; do not launch a long run yet')


## 4. Launch or resume your training

Run after the benchmark passes and the allocation covers the estimated duration.
Rerunning this cell checks the existing process instead of starting a duplicate.
If the process stopped and a verified completed-epoch checkpoint exists, this resumes
it with the same configuration. Do not change the config after starting this run.


In [ ]:
result=subprocess.run([sys.executable,'-u',str(SRC/'lab_tools.py'),'launch','--run-id',RUN_ID],cwd=REPO_ROOT)
if result.returncode:
    raise RuntimeError('Launch/resume failed; read the displayed console error')


## 5. Check progress — rerun this cell whenever needed


In [ ]:
subprocess.run([sys.executable,'-u',str(SRC/'lab_tools.py'),'status','--run-id',RUN_ID],cwd=REPO_ROOT,check=True)
subprocess.run(['nvidia-smi'],check=False)


## After training

Open `02_evaluate_viraat_cyclegan.ipynb` in the same runtime. The last complete
checkpoint and its checksum are saved after every epoch. Every tenth epoch is also
archived. Keep raw logs unchanged and copy checkpoints/evidence off the machine
before its allocation ends. No full training or new scores have been produced by
this starter package.
